In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
datatypes = {}
for col in ['fare_amount', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude']:
    datatypes[col] = 'float32'
for col in ['passenger_count']:
    datatypes[col] = 'uint8'

col_list = ['pickup_datetime','fare_amount', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'passenger_count']
col_list2 = ['pickup_datetime', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'passenger_count']

import random
random.seed(123)

train = pd.read_csv('/kaggle/input/nyc-taxi-fare-datasets/train.csv',
                    usecols = col_list, parse_dates = ['pickup_datetime'],
                    dtype = datatypes,
                    )
test = pd.read_csv('/kaggle/input/nyc-taxi-fare-datasets/test.csv', parse_dates = ['pickup_datetime'], dtype = datatypes)
train = train.dropna()

In [ ]:
!pip install pyarrow

#saves the full input dataset
train.to_parquet('train.parquet')

In [ ]:
#dictionary of parameters to filter the dataframe
filters = {'passenger_count': (1,6),
           'pickup_longitude': (-74,-72),
           'dropoff_longitude': (-74,-72),
           'pickup_latitude': (40,42),
           'dropoff_latitude': (40,42),
           'fare_amount': (2.5,100)
          }

#function that takes the dataframe and the filters and filters the data
def filter_df(df, filters):
    for column, condition in filters.items():
        if isinstance(condition,tuple):
            min_val, max_val = condition
            df = df[(df[column] >= min_val) & (df[column] <= max_val)]
        else:
            df = df[df[column] == condition]
    return df

start_time = train.pickup_datetime.min()

#function for creating a feature that measures the time from the earliest time in the dataframe
def time_elapsed(train):
    train['time_elapsed'] = (train.pickup_datetime - start_time).dt.total_seconds()

def year_and_hour(train):
    train['pickup_datetime'] = pd.to_datetime(train.pickup_datetime)
    train['year'] = train.pickup_datetime.dt.year
    train['hour'] = train.pickup_datetime.dt.hour
    train['peak_hour'] = train['hour'].apply(lambda x: 0 if not( 16 <= x <= 20) else 1) # for pickup times between 4pm and 8pm which is set to be the peak hour for rides and has a $5 surcharge

#function that gets the absolute value difference in pickup and dropoff distance
def abs_lat_lon(train):
    train['abs_lon_diff'] = np.abs(train.pickup_longitude - train.dropoff_longitude)
    train['abs_lat_diff'] = np.abs(train.pickup_latitude - train.dropoff_latitude)

#uses haversine formula to calculate the miles
def haversine(train):
    R = 3959
    # Convert latitude and longitude to radians
    lon1,lat1 = np.radians(train.pickup_longitude), np.radians(train.pickup_latitude)
    lon2,lat2 = np.radians(train.dropoff_longitude), np.radians(train.dropoff_latitude)

    # Find the differences
    dlon = lon2 - lon1
    dlat = lat2 - lat1

    # Apply the formula
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    # Calculate the angle (in radians)
    c = 2 * np.arcsin(np.sqrt(a))
    # Convert to miles
    mi = R * c

    train['haversine']  = mi

def haversine_airport(lat_airport, lon_airport, lat_dropoff, lon_dropoff):
    R = 3959

    lon1,lat1 = np.radians(lon_airport), np.radians(lat_airport)
    lon2,lat2 = np.radians(lon_dropoff), np.radians(lat_dropoff)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    # Convert to miles
    mi = R * c

    return mi

# to add locations for airports in ny
jfk_lat_lon = 40.641766, -73.780968
lga_lat_lon = 40.776863, -73.874069
ewr_lat_lon = 40.689491, -74.174538

#adds distance from a specific airport in NY to the dropoff location
def add_landmark_distance(df, landmark_name, landmark_lat_lon):
    lat, lon = landmark_lat_lon
    df[landmark_name+'_distance'] = haversine_airport(lat, lon, df.dropoff_latitude, df.dropoff_longitude)

def add_landmarks(df):
    landmarks = [('jfk',jfk_lat_lon),('lga',lga_lat_lon),('ewr',ewr_lat_lon)]
    for name, latlon in landmarks:
        add_landmark_distance(df, name, latlon)

#drop invalid/ unwanted distance data
filters_drop_haversine = [
                            lambda df: df['haversine'] < 0.01,
                            lambda df: (df['fare_amount'] > 2.5) & (df['haversine'] < 0.02),
                            lambda df: (df['fare_amount']< 2.6) & (df['haversine'] > 1),
                            lambda df: (df['fare_amount']> 80) & (df['haversine'] < 10.1),
                            lambda df: (df['fare_amount'] > 20) & (df['haversine'] < 0.2),
                            lambda df: (df['fare_amount'] > 60) & (df['haversine'] < 1),
                            lambda df: (df['fare_amount'] < 10) & (df['haversine'] > 25),
                            lambda df: (df['fare_amount'] > 19) & (df['haversine'] < 0.4),
                            lambda df: (df['jfk_distance'] < 1.5) & (df['haversine'] > 1) & (df['fare_amount'] < 10),
                            lambda df: (df['lga_distance'] <= 1.5) & (df['haversine'] > 0.03) & (df['fare_amount'] < 5)
                         ]

def drop_haversine(df, filters_drop_haversine):
    for condition in filters_drop_haversine:
        if callable(condition):
            mask = condition(df)
        else:
            mask = condition
        df = df.drop(df[mask].index)
    return df


def feature_engineering(train):
    year_and_hour(train)
    abs_lat_lon(train)
    haversine(train)
    add_landmarks(train)
    time_elapsed(train)

#filter the training data
train = filter_df(train, filters)
feature_engineering(train)
train = drop_haversine(train, filters_drop_haversine)

In [ ]:
#partitioning the data for train and testing and validation
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

input_col = ['pickup_longitude', 'pickup_latitude',
       'dropoff_longitude', 'dropoff_latitude', 'passenger_count', 'year',
       'hour', 'peak_hour', 'abs_lon_diff', 'abs_lat_diff', 'haversine',
       'jfk_distance', 'lga_distance', 'ewr_distance', 'time_elapsed']

train, val = train_test_split(train,test_size=0.2,random_state=123)
train_input = train[input_col]
train_target = train['fare_amount']
val_input = val[input_col]
val_target = val['fare_amount']
test = test[input_col]

In [ ]:
#saving the input data to load for other models
train_input.to_parquet('train_input.parquet')
val_input.to_parquet('val_input.parquet')
test.to_parquet('test.parquet')

In [ ]:
# when submitted to kaggele final score was 5.0098 so 1155 on leaderboard
from sklearn.linear_model import LinearRegression

model_lr = LinearRegression()
model_lr.fit(train_input, train_target)
val_pred = model_lr.predict(val_input)

rmse_lr =np.sqrt(mean_squared_error(val_target,val_pred))
print(rmse_lr)

submission_linear_reg = pd.read_csv('/kaggle/input/nyc-taxi-fare-datasets/sample_submission.csv')
test_pred = model_lr.predict(test)
submission_linear_reg['fare_amount'] = test_pred
submission_linear_reg.to_csv('linear_reg_sub.csv', index=None)


# model trained on 50% of the data. submission score was 5.27594
train2, val2 = train_test_split(train2,test_size=0.2,random_state=123)
train2_input = train2[input_col]
train2_target = train2['fare_amount']
val2_input = val2[input_col]
val2_target = val2['fare_amount']

model_lr2 = LinearRegression()
model_lr2.fit(train2_input, train2_target)
train2_pred = model_lr.predict(train2_input)

rmse_lr2 =np.sqrt(mean_squared_error(train2_target,train2_pred))
print(rmse_lr2)

submission_linear_reg2 = pd.read_csv('/kaggle/input/nyc-taxi-fare-datasets/sample_submission.csv')
test_pred2 = model_lr2.predict(test)
submission_linear_reg2['fare_amount'] = test_pred2
submission_linear_reg2.to_csv('linear_reg_sub2.csv', index=None)